# Mask-Guided Shadow Removal: Train · Validate · Test (Google Colab)

This notebook runs the whole project on Colab:

1. **Setup**: GPU check, mount Google Drive, install missing packages (Colab's own PyTorch is kept as is)
2. **Data**: download ISTD, look at sample triplets, check the scene-disjoint train/val/test split
3. **Configure**: hyper-parameters, mixed precision chosen automatically for the GPU
4. **Train + validate**: validation every N epochs, keeping the best checkpoint by PSNR
5. **Validation results**: learning curves, validation metrics, visual samples
6. **Test results**: PSNR / SSIM / MAE on the whole image, shadow region and non-shadow region, with GT and predicted masks
7. **Analysis**: results by shadow intensity and scene condition
8. **Visuals, inference on your own photos, download results**

> **Before you start:** `Runtime → Change runtime type → GPU` (T4 is fine; A100/L4 are faster).
> Upload the project folder to Google Drive, e.g. `MyDrive/Image-Shadow-Removal`. Avoid spaces in the path.

## 1. Setup

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total,driver_version --format=csv

In [ ]:
# ---- Paths: edit PROJECT_DIR if you uploaded the folder somewhere else ----
import os
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/Image-Shadow-Removal'   # uploaded project folder
OUTPUT_DIR  = f'{PROJECT_DIR}/outputs'                         # runs/checkpoints/results (kept on Drive)
DATA_DIR    = '/content/data'                                  # dataset on fast local disk

# If you uploaded a .zip instead of the folder, it is extracted automatically
if not os.path.isdir(PROJECT_DIR) and os.path.exists(PROJECT_DIR + '.zip'):
    !unzip -q "{PROJECT_DIR}.zip" -d "{os.path.dirname(PROJECT_DIR)}"

assert os.path.exists(f'{PROJECT_DIR}/train.py'), f'train.py not found in {PROJECT_DIR} - fix PROJECT_DIR'
assert ' ' not in PROJECT_DIR, 'Please use a path without spaces'
os.makedirs(OUTPUT_DIR, exist_ok=True)
%cd {PROJECT_DIR}
!ls

### Install dependencies
Colab already ships **torch + torchvision as a matched pair built for its CUDA version**. Reinstalling them, or downgrading numpy, is the usual cause of torch dependency errors on Colab. This cell only installs packages that are **missing** and never touches torch, torchvision or numpy.

In [ ]:
import importlib.util, subprocess, sys
needed = {'yaml': 'pyyaml', 'tqdm': 'tqdm', 'pandas': 'pandas', 'matplotlib': 'matplotlib',
          'PIL': 'pillow', 'tensorboard': 'tensorboard', 'gdown': 'gdown'}
missing = [pkg for mod, pkg in needed.items() if importlib.util.find_spec(mod) is None]
if missing:
    print('Installing:', missing)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
else:
    print('All Python dependencies already present')

# ISTD is a RAR5 archive: Colab's old p7zip `7z` fails on it ("Unsupported Method"), so install unrar.
# (If apt fails, download_istd.sh falls back to the official 7-Zip binary automatically.)
!apt-get -qq update > /dev/null 2>&1; apt-get -qq install -y unrar > /dev/null 2>&1
!which unrar && echo "unrar OK" || echo "unrar not installed - the script will use the 7-Zip fallback"
!python check_env.py

## 2. Data: ISTD

ISTD contains **1,330 training and 540 test** triplets: shadow image `*_A`, shadow mask `*_B`, shadow-free ground truth `*_C`.

- **Option A**: automatic download from Google Drive (sometimes blocked by Drive's quota).
- **Option B**: download `ISTD_Dataset.rar` once from the [ST-CGAN repo](https://github.com/DeepInsight-PCALab/ST-CGAN), put it on your Drive, and set `ISTD_ARCHIVE` below.

In [ ]:
ISTD_ARCHIVE = ''   # Option B: e.g. '/content/drive/MyDrive/datasets/ISTD_Dataset.rar'
SAVE_ARCHIVE_TO_DRIVE = True   # keep a copy of the downloaded archive on Drive for next sessions

DRIVE_COPY = '/content/drive/MyDrive/datasets/ISTD_Dataset.rar'
if not ISTD_ARCHIVE and os.path.exists(DRIVE_COPY):
    ISTD_ARCHIVE = DRIVE_COPY
    print('Using cached archive on Drive:', DRIVE_COPY)

# (an incomplete folder from an earlier failed extraction is detected and re-extracted)
!DATA_DIR={DATA_DIR} bash scripts/download_istd.sh {ISTD_ARCHIVE}

DATA_ROOT = f'{DATA_DIR}/ISTD_Dataset'
assert os.path.isdir(f'{DATA_ROOT}/train/train_A'), 'Dataset not found - see the options above'
if SAVE_ARCHIVE_TO_DRIVE and not ISTD_ARCHIVE and os.path.exists(f'{DATA_DIR}/ISTD_Dataset.rar'):
    os.makedirs(os.path.dirname(DRIVE_COPY), exist_ok=True)
    !cp {DATA_DIR}/ISTD_Dataset.rar {DRIVE_COPY}

In [ ]:
# ---- Look at the data: splits + sample triplets ----
import sys, random
sys.path.insert(0, f'{PROJECT_DIR}/src')
import numpy as np, matplotlib.pyplot as plt, pandas as pd, torch
from shadow_removal.utils import load_config
from shadow_removal.data import build_datasets, scene_id
from shadow_removal.analysis import image_attributes

base_cfg = load_config('configs/istd.yaml', [f'data.root={DATA_ROOT}'])
train_ds, val_ds, test_ds = build_datasets(base_cfg)
split_df = pd.DataFrame({
    'split': ['train', 'val', 'test'],
    'images': [len(train_ds), len(val_ds), len(test_ds)],
    'scenes': [len({scene_id(s['name']) for s in d.samples}) for d in (train_ds, val_ds, test_ds)]})
display(split_df)
overlap = {scene_id(s['name']) for s in train_ds.samples} & {scene_id(s['name']) for s in val_ds.samples}
print('Scenes shared by train and val:', len(overlap), '(should be 0)')

fig, axes = plt.subplots(3, 4, figsize=(16, 9))
for col, i in enumerate(random.Random(0).sample(range(len(test_ds)), 4)):
    s = test_ds[i]
    a = image_attributes(s['shadow'], s['gt'], s['mask'])
    for row, (key, title) in enumerate([('shadow', 'Shadow image'), ('mask', 'Shadow mask'), ('gt', 'Shadow-free GT')]):
        img = s[key].permute(1, 2, 0).numpy()
        axes[row, col].imshow(img.squeeze(), cmap='gray' if key == 'mask' else None)
        axes[row, col].set_title(f"{title}\n{s['name']}" if row == 0 else title, fontsize=9)
        axes[row, col].axis('off')
    axes[2, col].set_title(f"GT | intensity {a['shadow_intensity']:.2f}, area {a['shadow_area']:.2f}", fontsize=9)
plt.tight_layout(); plt.show()

## 3. Configure the experiment

In [ ]:
import shlex
GPU         = 0          # index of the CUDA device (Colab has one GPU -> 0)
EXP_NAME    = 'istd_maskguided_unet'
EPOCHS      = 200        # ~1.5-3 h on a T4; lower it if your session is short
BATCH_SIZE  = 8          # T4 (16 GB) handles 8-12 at 256x256
CROP_SIZE   = 256
LR          = 2e-4
VAL_EVERY   = 5          # run validation every N epochs
USE_DETECTOR = True      # also learn to predict shadow masks
PERCEPTUAL  = 0.05       # VGG16 perceptual loss weight (0 = off)
NUM_WORKERS = 2          # Colab has 2 CPU cores
QUICK_RUN   = False      # True -> 2-epoch smoke test to check everything works end to end

# Mixed precision: bf16 only on Ampere+ (A100/L4); fp16 on T4/V100
cc_major = torch.cuda.get_device_capability(GPU)[0] if torch.cuda.is_available() else 0
AMP = 'bf16' if cc_major >= 8 else ('fp16' if cc_major > 0 else 'none')

if QUICK_RUN:
    EPOCHS, VAL_EVERY = 2, 1

OPTS = [f'data.root={DATA_ROOT}', f'experiment.output_dir={OUTPUT_DIR}',
        f'train.epochs={EPOCHS}', f'train.batch_size={BATCH_SIZE}', f'data.crop_size={CROP_SIZE}',
        f'train.lr={LR}', f'train.val_every={VAL_EVERY}', f'train.amp={AMP}',
        f'model.detector.enabled={str(USE_DETECTOR).lower()}', f'loss.perceptual={PERCEPTUAL}',
        f'data.num_workers={NUM_WORKERS}', 'train.save_every=0']
OPTS_STR = ' '.join(shlex.quote(o) for o in OPTS)
print('GPU:', torch.cuda.get_device_name(GPU) if torch.cuda.is_available() else 'none', '| AMP:', AMP)
print(OPTS_STR)

In [ ]:
# ---- Model check: parameter count + one full-resolution forward pass ----
from shadow_removal.models import ShadowRemovalNet, count_parameters
cfg = load_config('configs/istd.yaml', OPTS)
model = ShadowRemovalNet.from_config(cfg['model']).cuda(GPU).eval()
print(f'Removal U-Net : {count_parameters(model.remover)/1e6:.2f} M params')
if model.has_detector:
    print(f'Detector U-Net: {count_parameters(model.detector)/1e6:.2f} M params')
with torch.no_grad():
    x = torch.rand(1, 3, 480, 640, device=f'cuda:{GPU}')
    m = (torch.rand(1, 1, 480, 640, device=f'cuda:{GPU}') > 0.7).float()
    out = model(x, m, 'gt')
print({k: tuple(v.shape) for k, v in out.items()})
print(f'GPU memory used: {torch.cuda.max_memory_allocated(GPU)/1024**3:.2f} GB')
del model, out; torch.cuda.empty_cache()

## 4. Train + validate
Each epoch is logged. Every `VAL_EVERY` epochs the model is scored on the **held-out validation scenes**, and the best checkpoint (by PSNR) is saved to `checkpoints/best.pth`. When training finishes, the **test set is evaluated automatically**, once with the GT masks and once with the predicted masks.

Everything is written to Drive under `OUTPUT_DIR`, so a disconnect loses at most one epoch. Use the *Resume* cell below to continue.

In [ ]:
!python train.py --config configs/istd.yaml --gpu {GPU} --exp-name {EXP_NAME} --opts {OPTS_STR}

### Resume after a disconnect (run only if needed)

In [ ]:
import glob
def latest_run(exp_name=EXP_NAME):
    runs = sorted(glob.glob(f'{OUTPUT_DIR}/{exp_name}_*'), key=os.path.getmtime)
    assert runs, f'No runs found in {OUTPUT_DIR}'
    return runs[-1]

LAST_CKPT = f'{latest_run()}/checkpoints/last.pth'
print('Resuming from', LAST_CKPT)
!python train.py --resume {LAST_CKPT} --gpu {GPU}

## 5. Validation results

In [ ]:
import glob, json
from IPython.display import Image, Markdown, display

def latest_run(exp_name=EXP_NAME):
    runs = sorted(glob.glob(f'{OUTPUT_DIR}/{exp_name}_*'), key=os.path.getmtime)
    assert runs, f'No runs found in {OUTPUT_DIR}'
    return runs[-1]

RUN_DIR = latest_run()          # or set a specific run folder by hand
print('Run:', RUN_DIR)
!tail -n 5 {RUN_DIR}/train.log
display(Image(f'{RUN_DIR}/training_curves.png'))

In [ ]:
hist = pd.read_csv(f'{RUN_DIR}/history.csv')
val_cols = ['epoch', 'val_psnr_all', 'val_psnr_shadow', 'val_psnr_nonshadow', 'val_ssim_all',
            'val_mae_lab_all', 'val_mae_lab_shadow', 'val_det_ber']
val_hist = hist[[c for c in val_cols if c in hist]].dropna(subset=['val_psnr_all'])
best = val_hist.loc[val_hist['val_psnr_all'].idxmax()]
print(f"Best validation epoch: {int(best['epoch'])} | PSNR {best['val_psnr_all']:.2f} dB | "
      f"SSIM {best['val_ssim_all']:.4f} | MAE-LAB {best['val_mae_lab_all']:.3f}")
display(val_hist.style.format(precision=3).highlight_max(subset=['val_psnr_all'], color='#cfe3fa'))

In [ ]:
# Validation samples from the best epoch:  input | GT mask | pred mask | output | GT | error x4
best_vis = sorted(glob.glob(f"{RUN_DIR}/val_vis/epoch_{int(best['epoch']):03d}/*.png"))
for p in best_vis:
    print(os.path.basename(p)); display(Image(p, width=1100))

### Full validation evaluation with the best checkpoint
Runs the complete evaluation (all metrics, regions and analysis) on the validation split. Results go to `results/val_gt`.

In [ ]:
BEST_CKPT = f'{RUN_DIR}/checkpoints/best.pth'
!python test.py --checkpoint {BEST_CKPT} --gpu {GPU} --split val --mask-source gt --num-vis 8 --no-save-images
print(json.dumps({k: round(v, 4) for k, v in json.load(open(f'{RUN_DIR}/results/val_gt/summary.json')).items()
                  if isinstance(v, float) and k.endswith(('_all', '_shadow', '_nonshadow'))}, indent=1))

## 6. Test results
The test set was already evaluated at the end of training. Set `RERUN_TEST = True` to evaluate again, for example with another checkpoint or the 256×256 protocol.

In [ ]:
RERUN_TEST = False
EXTRA = ''   # e.g. "--opts data.eval_size=[256,256]" for the 256x256 evaluation protocol
if RERUN_TEST or not os.path.exists(f'{RUN_DIR}/results/test_gt/summary.json'):
    !python test.py --checkpoint {BEST_CKPT} --gpu {GPU} --split test --mask-source both {EXTRA}

In [ ]:
REGIONS = [('all', 'Whole image'), ('shadow', 'Shadow region'), ('nonshadow', 'Non-shadow region')]
METRICS = [('psnr', 'PSNR ↑'), ('ssim', 'SSIM ↑'), ('mae_lab', 'MAE LAB ↓'), ('mae_rgb', 'MAE RGB ↓')]

def results_table(summary):
    rows = []
    for r, rname in REGIONS:
        for who, prefix in (('Input (shadow image)', 'input_'), ('Model output', '')):
            rows.append({'Region': rname, 'Source': who,
                         **{mname: summary.get(f'{prefix}{m}_{r}') for m, mname in METRICS}})
    return pd.DataFrame(rows).set_index(['Region', 'Source'])

for src in ('gt', 'pred'):
    p = f'{RUN_DIR}/results/test_{src}/summary.json'
    if not os.path.exists(p):
        continue
    s = json.load(open(p))
    display(Markdown(f"### Test set, {'GT' if src == 'gt' else 'predicted (detector)'} masks — "
                     f"{s['num_images']} images, {s['inference_ms_per_image']:.1f} ms/image"))
    display(results_table(s).style.format(precision=3))
    if 'det_ber' in s:
        print(f"Shadow detection: BER {s['det_ber']:.2f}% | IoU {s['det_iou']:.4f} | F1 {s['det_f1']:.4f}")
    print(f"PSNR gain over input: {s['psnr_gain_all']:+.2f} dB")

## 7. Analysis by shadow intensity and scene condition

In [ ]:
SRC = 'gt'   # 'gt' or 'pred'
A = f'{RUN_DIR}/results/test_{SRC}/analysis'
print('Bin edges (tertiles):', json.load(open(f'{A}/bin_edges.json')))
cols = ['bin', 'n', 'input_psnr_all', 'psnr_all', 'psnr_shadow', 'input_ssim_all', 'ssim_all',
        'input_mae_lab_all', 'mae_lab_all']
for attr in ['shadow_intensity', 'shadow_area', 'scene_brightness', 'scene_texture']:
    f = f'{A}/by_{attr}.csv'
    if os.path.exists(f):
        display(Markdown(f"#### {attr.replace('_', ' ').title()}"))
        t = pd.read_csv(f)
        display(t[[c for c in cols if c in t]].style.format(precision=3).hide(axis='index'))
        display(Image(f'{A}/plots/by_{attr}.png', width=1000))
display(Image(f'{A}/plots/psnr_distribution.png', width=1000))

In [ ]:
# Per-image and per-scene results
per_img = pd.read_csv(f'{RUN_DIR}/results/test_{SRC}/per_image_metrics.csv')
per_img['psnr_gain'] = per_img['psnr_all'] - per_img['input_psnr_all']
show = ['name', 'psnr_all', 'input_psnr_all', 'psnr_gain', 'ssim_all', 'mae_lab_all', 'shadow_intensity', 'shadow_area']
display(Markdown('**Hardest 10 test images (lowest PSNR)**'))
display(per_img.nsmallest(10, 'psnr_all')[show].style.format(precision=3).hide(axis='index'))
per_scene = pd.read_csv(f'{RUN_DIR}/results/test_{SRC}/per_scene_metrics.csv')
display(Markdown(f'**Per-scene PSNR** ({len(per_scene)} scenes)'))
display(per_scene[['scene', 'n', 'psnr_all', 'ssim_all', 'mae_lab_all']].sort_values('psnr_all')
        .style.format(precision=3).hide(axis='index'))

## 8. Visual results

In [ ]:
# Best and worst test images:  input | GT mask | (pred mask) | output | GT | |error| x4
R = f'{RUN_DIR}/results/test_{SRC}'
for tag in ('best', 'worst'):
    display(Markdown(f'### {tag.title()} cases'))
    for p in sorted(glob.glob(f'{R}/extremes/{tag}_*.png'))[:3]:
        print(os.path.basename(p)); display(Image(p, width=1100))

In [ ]:
# More comparisons (first N test images)
for p in sorted(glob.glob(f'{R}/comparisons/*.png'))[:6]:
    print(os.path.basename(p)); display(Image(p, width=1100))

### TensorBoard (optional)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {RUN_DIR}/tensorboard

## 9. Try it on your own photos
Upload images (and optionally masks with the same file names). Without masks, the detector branch predicts them.

In [ ]:
from google.colab import files
MY_DIR = '/content/my_images'; MY_OUT = '/content/my_results'
os.makedirs(MY_DIR, exist_ok=True)
for name, data in files.upload().items():
    open(f'{MY_DIR}/{name}', 'wb').write(data)
!python infer.py --checkpoint {BEST_CKPT} --input {MY_DIR} --output {MY_OUT} --gpu {GPU} --max-side 1024 --save-comparison
for p in sorted(glob.glob(f'{MY_OUT}/*_comparison.png')):
    display(Image(p, width=1100))

## 10. Download results
Packs config, logs, curves, `best.pth` and the whole `results/` folder into one zip. Everything also stays on Drive under `OUTPUT_DIR`.

In [ ]:
import shutil
ZIP_BASE = f'/content/{os.path.basename(RUN_DIR)}_results'
stage = '/content/_stage'; shutil.rmtree(stage, ignore_errors=True)
shutil.copytree(RUN_DIR, stage, ignore=shutil.ignore_patterns('last.pth', 'epoch_*.pth', 'tensorboard'))
zip_path = shutil.make_archive(ZIP_BASE, 'zip', stage)
print(f'{zip_path}: {os.path.getsize(zip_path)/1024**2:.1f} MB')
files.download(zip_path)